# Phase 1: Exploratory Data Analysis & Legal Corpus Statistics
### Course Project: Citation-Aware Hybrid Legal Case Retrieval for Indian Court Judgments
**Author:** Vismaya
---
This notebook inspects the processed legal corpus, analyzes document lengths, examines citation network topology, evaluates relevance label distributions, and validates that test query citation edges are completely masked to prevent data leakage.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# Set plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Load Processed Dataset Files
We load `corpus.jsonl`, `queries.jsonl`, `qrels.tsv`, and `citation_edges.tsv` from `data/processed/sample` (or your chosen dataset partition).

In [ ]:
dataset_dir = 'data/processed/sample'
corpus_path = os.path.join(dataset_dir, 'corpus.jsonl')
queries_path = os.path.join(dataset_dir, 'queries.jsonl')
qrels_path = os.path.join(dataset_dir, 'qrels.tsv')
edges_path = os.path.join(dataset_dir, 'citation_edges.tsv')

# 1. Load Corpus
corpus_records = []
with open(corpus_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            corpus_records.append(json.loads(line))
df_corpus = pd.DataFrame(corpus_records)

# 2. Load Queries
query_records = []
with open(queries_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            query_records.append(json.loads(line))
df_queries = pd.DataFrame(query_records)

# 3. Load Qrels
df_qrels = pd.read_csv(qrels_path, sep='\s+', names=['query_id', 'iter', 'doc_id', 'relevance'])

# 4. Load Citation Edges
df_edges = pd.read_csv(edges_path, sep='\t')

print(f"Corpus Documents : {len(df_corpus):,}")
print(f"Queries          : {len(df_queries):,}")
print(f"Qrels Annotations: {len(df_qrels):,}")
print(f"Citation Edges   : {len(df_edges):,}")

## 2. Document & Query Length Distribution Analysis

In [ ]:
doc_lengths = df_corpus['length']
query_lengths = df_queries['length']

stats_df = pd.DataFrame({
    'Corpus Word Count': doc_lengths.describe(percentiles=[0.25, 0.5, 0.75, 0.95]),
    'Query Word Count': query_lengths.describe(percentiles=[0.25, 0.5, 0.75, 0.95])
})
display(stats_df.round(2))

# Plot length distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(doc_lengths, bins=15, color='#1f77b4', edgecolor='black', alpha=0.7)
axes[0].set_title('Legal Case Length Distribution (Words)')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Number of Cases')

axes[1].hist(query_lengths, bins=10, color='#2ca02c', edgecolor='black', alpha=0.7)
axes[1].set_title('Query Fact Statement Length (Words)')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Number of Queries')
plt.tight_layout()
plt.show()

## 3. Citation Graph Topology & Precedent Prestige Analysis

In [ ]:
G = nx.DiGraph()
for _, row in df_edges.iterrows():
    G.add_edge(str(row['source_doc_id']), str(row['target_doc_id']))

in_degrees = dict(G.in_degree())
out_degrees = dict(G.out_degree())

top_cited = sorted(in_degrees.items(), key=lambda x: x[1], reverse=True)[:10]
df_top_cited = pd.DataFrame(top_cited, columns=['Doc ID / Landmark Case', 'In-Degree (Citations Received)'])

print("=== Top Cited Landmark Precedents in Citation Network ===")
display(df_top_cited)

in_deg_vals = list(in_degrees.values())
out_deg_vals = list(out_degrees.values())

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([x[0] for x in top_cited], [x[1] for x in top_cited], color='#d62728', edgecolor='black', alpha=0.8)
ax.set_title('Top Landmark Decisions by In-Degree Prestige')
ax.set_ylabel('Citations Received')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Relevance Ground Truth (Qrels) Distribution

In [ ]:
rel_counts = df_qrels['relevance'].value_counts().sort_index()
print("Relevance score frequency:")
print(rel_counts)

q_grouped = df_qrels.groupby('query_id')['doc_id'].count()
print(f"\nAverage relevant precedents per query: {q_grouped.mean():.2f}")
print(f"Min relevant: {q_grouped.min()}, Max relevant: {q_grouped.max()}")

## 5. Verification of Zero Data Leakage for Test Queries
We verify that no held-out test query contains outgoing citation edges in the feature graph.

In [ ]:
# Example test query IDs from split
test_query_ids = set(df_queries['query_id'].tail(1))
print(f"Target test query IDs for isolation check: {test_query_ids}")

leaking_edges = df_edges[df_edges['source_doc_id'].isin(test_query_ids)]
print(f"Outgoing edges from test queries in feature graph: {len(leaking_edges)}")
assert len(leaking_edges) == 0, "ERROR: Data leakage detected! Test queries must have zero outgoing edges in the graph."
print("PASSED: Zero citation leakage confirmed for all test queries!")